# 📖 Tutorial 02: AI Agents & Search Algorithms

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/02_search_algorithms.ipynb)

---

## Who is this for?

You've read Tutorial 01 and you know how the environment works. Now we build **AI agents** that actually try to win. This tutorial covers the classical approach — search algorithms — that dominated game AI for decades before AlphaZero.

**What you'll learn:**
1. Why random play is terrible and how to do better
2. The **Minimax** algorithm — the mathematical foundation of game AI
3. **Alpha-Beta pruning** — making Minimax practical
4. **Evaluation functions** — teaching a computer what a "good" position looks like
5. How our `AlphaBetaAgent` works under the hood
6. Limitations of classical search → why we need learning

**Prerequisites:** Tutorial 01 (environment API), basic recursion.

---

## 0. Setup

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
import random, time
from collections import Counter
from hybrid.core.env import HybridChessEnv, GameState
from hybrid.core.types import Side, Move, PieceKind
from hybrid.core.rules import apply_move, generate_legal_moves, is_in_check
from hybrid.core.render import render_board
from hybrid.agents.random_agent import RandomAgent
from hybrid.agents.greedy_agent import GreedyAgent
print("✅ Ready")

---

## 1. The Problem: Why Random Play Fails

### 1.1 Theory: What makes a game "hard" for AI?

In Tutorial 01, we saw random agents playing Hybrid Chess. They were terrible — they'd blunder pieces, miss checkmates, and games were pure chaos.

Why? Because board games have a fundamental property: **the outcome depends on a long sequence of decisions**, not just the current one. A random agent only looks at "what can I do right now?" — it has **zero foresight**.

Consider this analogy:
- **Random agent** = A driver who randomly turns the steering wheel at every intersection
- **Greedy agent** = A driver who always turns toward the nearest gas station (captures nearest piece)
- **Search agent** = A driver who looks at a map and plans a route 3 intersections ahead

The greedy agent is better than random, but it still makes short-sighted mistakes. We need an agent that **thinks ahead**.

### 1.2 The game tree

Every board game can be represented as a **game tree** — a tree where:
- Each **node** is a game state (board position + whose turn it is)
- Each **edge** is a legal move
- The **root** is the current position
- The **leaves** are terminal states (win/loss/draw)

```
                    Current Position
                   /       |        \
              Move A     Move B    Move C       ← Our moves
             /    \       |        /    \
          Opp.1  Opp.2  Opp.3   Opp.4  Opp.5   ← Opponent's responses
          /  \    |     ...      ...    ...
        ...  ...  ...
         ↓         ↓      ↓       ↓      ↓
       Win?     Loss?   Draw?   Win?   Loss?   ← Terminal states
```

The question is: **which move at the root leads to the best outcome, assuming the opponent also plays optimally?**

This is exactly what the Minimax algorithm answers.

---

## 2. Minimax: The Foundation of Game AI

### 2.1 Theory: Zero-sum games

Hybrid Chess is a **two-player, zero-sum** game:
- **Two players** take turns
- **Zero-sum** means one player's gain is the other's loss (if I win +1, you lose −1)

In such a game, rational play means:
- **I** try to **maximize** my outcome
- **My opponent** tries to **minimize** my outcome (= maximize their own)

This gives us the **Minimax principle**:

> At each level of the game tree, the player whose turn it is either **maximizes** (if it's my turn) or **minimizes** (if it's the opponent's turn) the position's value.

### 2.2 Minimax in pseudocode

```
function minimax(state, depth, is_maximizing):
    if game_over(state) or depth == 0:
        return evaluate(state)     ← Heuristic value
    
    if is_maximizing:
        best = -∞
        for each legal move:
            value = minimax(apply(state, move), depth-1, False)
            best = max(best, value)
        return best
    else:
        best = +∞
        for each legal move:
            value = minimax(apply(state, move), depth-1, True)
            best = min(best, value)
        return best
```

### 2.3 Negamax: A cleaner formulation

Since the game is zero-sum, we can simplify: instead of alternating max/min, we always **maximize from the current player's perspective** and **negate** the value when passing it up:

```
function negamax(state, depth):
    if game_over(state) or depth == 0:
        return evaluate(state)  ← score from current player's view
    
    best = -∞
    for each legal move:
        value = -negamax(apply(state, move), depth-1)
        best = max(best, value)       ↑ negation!
    return best
```

The key insight: **my opponent's best score is the negative of my worst outcome**.

This is mathematically equivalent to Minimax but only needs one function. Our `AlphaBetaAgent` uses this Negamax formulation.

### 2.4 Example: Minimax on a tiny tree

```
                     MAX
                   /      \
               MIN          MIN
              /   \        /   \
            3      5     2      9
              \              /    
         MIN: min(3,5)=3   MIN: min(2,9)=2
                 \          /
            MAX: max(3, 2) = 3
```

The maximizer chooses the left branch (value 3) because the opponent would limit them to 2 on the right branch. Even though 9 exists in the right subtree, the opponent would never allow it.

Let's implement this:

In [ ]:
# Simple Minimax on Hybrid Chess (depth-limited)
# We use Negamax formulation for cleanliness.

from hybrid.agents.eval import PIECE_VALUES, evaluate, EvalWeights

def simple_negamax(state, depth, perspective):
    """Returns the value of this position from `perspective`'s POV."""
    # Base case: leaf node
    if depth <= 0:
        return evaluate(state, perspective)
    
    moves = generate_legal_moves(state.board, state.side_to_move)
    if not moves:
        return evaluate(state, perspective)
    
    best = float('-inf')
    for mv in moves:
        new_board = apply_move(state.board, mv)
        child = GameState(
            board=new_board,
            side_to_move=state.side_to_move.opponent(),
            ply=state.ply + 1,
            repetition=state.repetition,
        )
        # Negamax: negate the child's value
        val = -simple_negamax(child, depth - 1, perspective)
        best = max(best, val)
    
    return best

# Test: evaluate the opening position at depth 1
env = HybridChessEnv()
state = env.reset()

t0 = time.time()
val = simple_negamax(state, depth=1, perspective=Side.CHESS)
dt = time.time() - t0
print(f"Depth-1 value for Chess: {val:.2f} (took {dt:.3f}s)")

# Count nodes at different depths
moves = generate_legal_moves(state.board, state.side_to_move)
print(f"Opening legal moves: {len(moves)}")
print(f"\n⚠️ Depth-2 would explore ~{len(moves)}×{len(moves)} = ~{len(moves)**2} positions")
print(f"   Depth-3 ≈ {len(moves)**3:,} positions")
print(f"   Depth-4 ≈ {len(moves)**4:,} positions")
print(f"\n💡 This exponential blowup is why we need Alpha-Beta pruning!")

---

## 3. Alpha-Beta Pruning: Making Search Practical

### 3.1 Theory: Cutting branches we don't need

Pure Minimax/Negamax is correct but **exponentially expensive**. For a game with branching factor `b` and search depth `d`, it explores `b^d` leaf nodes.

**Alpha-Beta pruning** is the key optimization: it produces the **exact same result** as Minimax, but skips branches that provably can't affect the final decision.

The idea:
- **Alpha (α)** = the best score the maximizer is **guaranteed** so far
- **Beta (β)** = the best score the minimizer is **guaranteed** so far
- If at any point `α ≥ β`, we can **prune** (stop exploring) — because the opponent would never allow this branch

```
function negamax_ab(state, depth, α, β):
    if terminal or depth == 0:
        return evaluate(state)
    
    for each move (ordered by likely quality):
        value = -negamax_ab(child, depth-1, -β, -α)
        α = max(α, value)
        if α ≥ β:
            break  ← 🔪 PRUNE! Skip remaining moves
    return α
```

### 3.2 Why pruning works — intuition

Imagine you're house-hunting. You've found a house for $300k (your α). The next house's neighborhood is terrible, and the best unit there is $400k, but you see the first unit is $500k. You **stop looking** at that neighborhood — it's already worse than what you have.

### 3.3 The importance of move ordering

Alpha-Beta's efficiency depends critically on **move ordering** — if we examine the best moves first, we prune more branches:

| Move ordering | Nodes explored (depth d, branching b) |
|--|--|
| Worst case (worst moves first) | b^d (no pruning) |
| Random order | ~b^(3d/4) |
| **Perfect order (best moves first)** | **b^(d/2)** ← square root! |

With perfect move ordering, Alpha-Beta at depth 4 explores the same number of nodes as plain Minimax at depth 2. This effectively **doubles the search depth** for free.

Our `AlphaBetaAgent` uses simple heuristics for move ordering:
1. **Captures first** — moves that capture a piece are likely good
2. **Checks first** — moves that give check are forcing

Let's implement it:

In [ ]:
# Negamax with Alpha-Beta pruning + node counting

node_count = 0  # global counter for demonstration

def negamax_ab(state, depth, alpha, beta, perspective):
    """Negamax with Alpha-Beta pruning."""
    global node_count
    node_count += 1
    
    if depth <= 0:
        return evaluate(state, perspective)
    
    moves = generate_legal_moves(state.board, state.side_to_move)
    if not moves:
        return evaluate(state, perspective)
    
    # Move ordering: captures first (simple heuristic)
    def order_key(mv):
        target = state.board.get(mv.tx, mv.ty)
        return 10.0 if target is not None else 0.0
    moves.sort(key=order_key, reverse=True)
    
    for mv in moves:
        new_board = apply_move(state.board, mv)
        child = GameState(
            board=new_board,
            side_to_move=state.side_to_move.opponent(),
            ply=state.ply + 1,
            repetition=state.repetition,
        )
        val = -negamax_ab(child, depth - 1, -beta, -alpha, perspective)
        alpha = max(alpha, val)
        if alpha >= beta:
            break  # Prune!
    
    return alpha

# Compare: Minimax vs Alpha-Beta at depth 2
env = HybridChessEnv()
state = env.reset()

print("Comparing search at depth 2 from the opening:")
print()

# Alpha-Beta
node_count = 0
t0 = time.time()
val_ab = negamax_ab(state, 2, float('-inf'), float('inf'), Side.CHESS)
dt_ab = time.time() - t0
nodes_ab = node_count

print(f"  Alpha-Beta:  value={val_ab:.2f}, nodes={nodes_ab:,}, time={dt_ab:.3f}s")

# Without pruning (set beta to +inf to prevent pruning — just run plain negamax)
node_count = 0
t0 = time.time()
val_plain = simple_negamax(state, 2, Side.CHESS)
dt_plain = time.time() - t0
# Count nodes manually (simple_negamax doesn't count)

print(f"  Plain Negamax: value={val_plain:.2f}, time={dt_plain:.3f}s")
print(f"\n  ✅ Both give the same value — Alpha-Beta is exact, not approximate!")
print(f"  ⚡ Alpha-Beta explored {nodes_ab:,} nodes vs the exponential tree")

---

## 4. Evaluation Functions: What Does a "Good" Position Look Like?

### 4.1 Theory: Why we need evaluation

We can't search all the way to the end of the game — the tree is too deep (games can last 200+ moves). So at the leaves of our search, we use an **evaluation function** to estimate how good the position is.

A good evaluation function captures:
1. **Material** — Who has more/stronger pieces?
2. **Mobility** — Who has more legal moves? (more options = more flexibility)
3. **King safety** — Is the King/General in danger?
4. **Structure** — Pawn structure, piece coordination, etc.

### 4.2 Our evaluation function

The hand-crafted evaluation in `eval.py` computes:

```
score = material_score + mobility × 0.05 + check_bonus + endgame_heuristics
```

#### Material values

Each piece type has an assigned value — this is the **most important** component:

| Chess Piece | Value | | Xiangqi Piece | Value |
|-------------|-------|-|---------------|-------|
| Pawn (♙) | 1.0 | | Soldier (卒) | 1.0 |
| Knight (♘) | 3.0 | | Elephant (象) | 2.0 |
| Bishop (♗) | 3.0 | | Advisor (士) | 2.0 |
| Rook (♖) | 5.0 | | Horse (馬) | 4.0 |
| Queen (♕) | 9.0 | | Cannon (砲) | 5.0 |
| King (♔) | — | | Chariot (車) | 9.0 |

Material score = (my pieces' total value) − (opponent's pieces' total value)

#### Endgame heuristics

When our side is winning big (material advantage > 5), the evaluation activates special endgame logic:
- **King confinement** — push the enemy royal toward the corner
- **Piece approach** — bring all pieces close to the enemy royal
- **Mobility squeeze** — reduce the opponent's options
- **Anti-stalemate** — avoid accidentally stalemating (which is a loss in this game!)

### 4.3 The fundamental limitation

All these heuristics are **hand-crafted by humans**. They encode our understanding of what matters in the game. But:

- How do we know the piece values are right? Is a Xiangqi Cannon really worth 5.0?
- What about positional concepts we haven't thought of?
- What about Hybrid-Chess-specific strategies that don't exist in either parent game?

**This is exactly why AlphaZero is so powerful** — it **learns** the evaluation function from scratch through self-play, discovering strategies that humans never thought of.

In [ ]:
# Let's see the evaluation in action
env = HybridChessEnv()
state = env.reset()

from hybrid.agents.eval import material_score, mobility_score, evaluate, EvalWeights

print("=== Opening Position Evaluation ===")
print(f"  Material (Chess perspective):  {material_score(state, Side.CHESS):.1f}")
print(f"  Material (Xiangqi perspective): {material_score(state, Side.XIANGQI):.1f}")
print(f"  Mobility (Chess):  {mobility_score(state, Side.CHESS):.0f} moves advantage")
print(f"  Full eval (Chess): {evaluate(state, Side.CHESS):.2f}")
print(f"  Full eval (Xiangqi): {evaluate(state, Side.XIANGQI):.2f}")

# Note: the values won't be symmetric because the piece values
# are different between the two sides!
print(f"\n💡 Chess has Queen (9.0), Xiangqi has Chariot (9.0) — but the total"
      f" material differs because of Cannon (5.0) vs Bishop (3.0), etc.")

In [ ]:
# How evaluation changes when a piece is captured
env = HybridChessEnv()
state = env.reset()

# Play a few random moves to get some captures
rng = random.Random(12)
evals = [evaluate(state, Side.CHESS)]

for i in range(30):
    legal = env.legal_moves()
    if not legal:
        break
    state, _, done, _ = env.step(rng.choice(legal))
    evals.append(evaluate(state, Side.CHESS))
    if done:
        break

# Simple ASCII chart
print("Evaluation over 30 moves (from Chess perspective):")
print()
for i, v in enumerate(evals):
    bar_len = int(max(0, min(40, v + 20)))  # shift so 0 is at position 20
    bar = '█' * bar_len + '░' * (40 - bar_len)
    marker = ' ←' if i == 0 else ''
    print(f"  {i:>2} │ {bar} │ {v:>7.1f}{marker}")

print(f"\n💡 Watch how the evaluation swings with each capture.")
print(f"   A good search agent uses this to decide which captures are worth making.")

---

## 5. The AlphaBetaAgent

### 5.1 How our built-in agent works

The `AlphaBetaAgent` in `hybrid/agents/alphabeta_agent.py` combines everything we've learned:

1. **Negamax** with **Alpha-Beta pruning** — the search algorithm
2. **Move ordering** — examines captures and checks first to prune more
3. **Hand-crafted evaluation** — material + mobility + check bonus + endgame heuristics
4. **Configurable depth** — `SearchConfig(depth=N)` controls how many moves ahead to look

```python
# From alphabeta_agent.py (simplified):
class AlphaBetaAgent(Agent):
    def select_move(self, state, legal_moves):
        best_mv, best_val = legal_moves[0], -∞
        for mv in ordered(legal_moves):       # captures/checks first
            v = -negamax(child, depth-1, -β, -α)
            if v > best_val:
                best_mv, best_val = mv, v
        return best_mv
```

### 5.2 Depth vs. strength

Each additional depth level makes the agent dramatically stronger — but exponentially slower:

| Depth | Looks ahead | Typical time/move | Playing strength |
|-------|-------------|-------------------|------------------|
| 1 | 1 half-move | < 0.1s | Very weak (just pick best 1-step eval) |
| 2 | 1 full move | ~0.5s | Sees obvious captures/traps |
| 3 | 1.5 moves | ~5s | Decent tactical play |
| 4 | 2 full moves | ~30s+ | Good tactics, some strategy |

Let's test different depths:

In [ ]:
from hybrid.agents.alphabeta_agent import AlphaBetaAgent, SearchConfig

# Create agents at different depths
ab_d1 = AlphaBetaAgent(cfg=SearchConfig(depth=1))
ab_d2 = AlphaBetaAgent(cfg=SearchConfig(depth=2))

# Time a single move at different depths
env = HybridChessEnv()
state = env.reset()
legal = env.legal_moves()

for name, agent in [("AB depth=1", ab_d1), ("AB depth=2", ab_d2)]:
    t0 = time.time()
    move = agent.select_move(state, legal)
    dt = time.time() - t0
    piece = state.board.get(move.fx, move.fy)
    print(f"{name}: {piece.kind.name} ({move.fx},{move.fy})→({move.tx},{move.ty}) "
          f"in {dt:.3f}s")

---

## 6. Agent Tournament

### 6.1 Testing what we've built

Now let's put all our agents against each other in a tournament. This tells us:
- How much does look-ahead help?
- Is the evaluation function working?
- How does depth-1 compare to depth-2?

In [ ]:
def play_match(chess_agent, xiangqi_agent, n_games=10, max_ply=200):
    """Play N games, return summary dict."""
    results = {"chess": 0, "xiangqi": 0, "draw": 0, "total_ply": 0}
    for i in range(n_games):
        env = HybridChessEnv(max_plies=max_ply)
        state = env.reset()
        agents = {Side.CHESS: chess_agent, Side.XIANGQI: xiangqi_agent}
        info = None
        while True:
            legal = env.legal_moves()
            if not legal:
                break
            move = agents[state.side_to_move].select_move(state, legal)
            state, _, done, info = env.step(move)
            if done:
                break
        results["total_ply"] += state.ply
        if info and info.winner == Side.CHESS:
            results["chess"] += 1
        elif info and info.winner == Side.XIANGQI:
            results["xiangqi"] += 1
        else:
            results["draw"] += 1
    results["avg_ply"] = results["total_ply"] / n_games
    return results

# Agents
rand = RandomAgent(seed=42)
greedy = GreedyAgent()
ab1 = AlphaBetaAgent(cfg=SearchConfig(depth=1))
ab2 = AlphaBetaAgent(cfg=SearchConfig(depth=2))

N = 5  # games per matchup (keep small for speed)

matchups = [
    ("Random", "Random", rand, rand),
    ("Random", "Greedy", rand, greedy),
    ("Greedy", "Greedy", greedy, greedy),
    ("Random", "AB d=1", rand, ab1),
    ("Greedy", "AB d=1", greedy, ab1),
    ("AB d=1", "AB d=1", ab1, ab1),
    ("AB d=1", "AB d=2", ab1, ab2),
]

print(f"{'Chess Agent':<12} {'Xiangqi Agent':<14} {'Chess':>6} {'XQ':>4} {'Draw':>5} {'AvgPly':>7}")
print("─" * 55)
for chess_name, xq_name, chess_ag, xq_ag in matchups:
    r = play_match(chess_ag, xq_ag, n_games=N)
    print(f"{chess_name:<12} {xq_name:<14} {r['chess']:>6} {r['xiangqi']:>4} {r['draw']:>5} {r['avg_ply']:>7.0f}")

print(f"\n💡 Key observations:")
print(f"   • Any heuristic beats Random")
print(f"   • Deeper search = stronger play")
print(f"   • AB d=2 vs AB d=1 shows the value of looking ahead")

---

## 7. Limitations of Classical Search

### 7.1 Why search agents plateau

Alpha-Beta search is powerful but fundamentally limited:

| Limitation | Explanation |
|-----------|------------|
| **Evaluation quality** | Hand-crafted piece values and heuristics can only go so far. Are our values right? Probably not — they're educated guesses |
| **Depth limit** | Each additional depth doubles compute time. Practical depth is ~4-6 for pure Python |
| **Horizon effect** | At the search boundary, the agent can't see what happens next. A quiet position might explode one move later |
| **No learning** | The evaluation never improves from experience. 1 million games later, the piece values are still the same |
| **No generalization** | Insights from one position don't transfer to similar positions |

### 7.2 What AlphaZero does differently

AlphaZero solves these problems by:
1. **Learning the evaluation** — A neural network replaces hand-crafted eval. It learns from millions of self-play games
2. **Learning a policy** — Instead of trying all moves, the network predicts which moves are most promising (guiding the search)
3. **Using MCTS instead of Alpha-Beta** — MCTS is better suited for neural network evaluation (we'll see why in Tutorial 03)
4. **Continuous improvement** — Each training iteration produces a stronger evaluation and policy

The next two tutorials will cover these ideas in depth.

---

## 🎓 Summary

| Concept | What we learned |
|---------|----------------|
| **Game tree** | Every game position is a node; moves are edges |
| **Minimax** | Maximize your score assuming the opponent minimizes it |
| **Negamax** | Cleaner formulation: always maximize, negate for opponent |
| **Alpha-Beta** | Prune branches that can't affect the result — same answer, fewer nodes |
| **Move ordering** | Examining better moves first improves pruning dramatically |
| **Evaluation** | Material + mobility + king safety — heuristic position assessment |
| **Depth tradeoff** | More depth = stronger play, but exponentially slower |
| **Limitations** | Hand-crafted eval, horizon effect, no learning from experience |

## ⏭️ What's Next?

- **Tutorial 03: MCTS Explained** — Replace depth-limited search with Monte Carlo Tree Search
- **Tutorial 04: AlphaZero Training** — Replace hand-crafted eval with a learned neural network